# Part 10: Summary Chart

### Ramith Vaid | NTU Singapore

This notebook draws the consolidated chart for the volatility project from the CSV files each part saved, and prints one line of key numbers per part. It runs no models. The written summary is in `SUMMARY.md`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print('Libraries loaded.')

## 1. Load the saved tables

In [ ]:
p2cov = pd.read_csv('part2_var_coverage.csv')
p3 = pd.read_csv('part3_block_coverage.csv')
p3pool = pd.read_csv('part3_pooled_coverage.csv')
p4 = pd.read_csv('part4_pooled.csv')
p4b = pd.read_csv('part4_by_block.csv')
p4re = pd.read_csv('part4_part1_rerun.csv')
p5 = pd.read_csv('part5_fit_comparison.csv')
p5oos = pd.read_csv('part5_oos_21day.csv')
p6 = pd.read_csv('part6_rmse_ratio_to_garch.csv', index_col=0)
p7 = pd.read_csv('part7_ensemble_six_assets_ratio.csv', index_col=0)
p8 = pd.read_csv('part8_by_horizon.csv')
print('Loaded.')

## 2. One line per part

In [ ]:
lines = []
pub = p4re[p4re.version.str.contains('published')].set_index('ticker')['garch_over_ml']
cor = p4re[p4re.version.str.contains('embargo')].set_index('ticker')['garch_over_ml']
lines.append(('1/4', 'GARCH/ML RMSE ratio, as published vs. corrected target', ', '.join(f"{t}: {pub[t]:.2f} -> {cor[t]:.2f}" for t in pub.index)))
r = p3pool[(p3pool.level == 0.99)]
lines.append(('2-3', 'VaR 99% pooled breach rate (2,250 days)', ', '.join(f"{a} {b}: {d:.2f}%" for a, b, d in zip(r.ticker, r.model, r['breach_rate_%']))))
lines.append(('4', 'GARCH beats ML, assets pooled', f"{int((p4['garch_over_ml'] < 1).sum())} of {len(p4)}; median ratio {p4['garch_over_ml'].median():.3f}"))
lines.append(('5', 'GJR beats GARCH on AIC', f"{int((p5['dAIC_gjr'] < 0).sum())} of {len(p5)}; EGARCH {int((p5['dAIC_egarch'] < 0).sum())} of {len(p5)}"))
lines.append(('6', 'mean RMSE ratio to GARCH: Ridge / GRU / GBR rich', f"{p6['ridge_rich'].mean():.3f} / {p6['gru'].mean():.3f} / {p6['gbr_rich'].mean():.3f}"))
lines.append(('7', 'mean RMSE ratio, GARCH + Ridge + GRU average', f"{p7['garch + ridge + gru'].mean():.3f}"))
h = p8.groupby('horizon')[['naive_over_garch', 'har_over_garch', 'gbr_over_garch']].mean()
lines.append(('8', 'mean ratio to GARCH at 5/10/21 days (HAR)', ' / '.join(f"{h.loc[k, 'har_over_garch']:.3f}" for k in (5, 10, 21))))
for part, what, val in lines:
    print(f'Part {part:<4} {what:<52} {val}')

## 3. Chart

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 10.5))
fig.patch.set_facecolor('#0d1117')
for ax in axes.ravel():
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#374151')
CY, OR, PU, GR, GY = '#00e5ff', '#f59e0b', '#a78bfa', '#34d399', '#9ca3af'

# (a) VaR 99%
ax = axes[0, 0]
sub = p3[p3.level == 0.99]
groups = [(t, b) for t in ['JPM', '^GSPC'] for b in ['block 1', 'block 2', 'block 3']]
x = np.arange(len(groups))
w = 0.36
for off, m, col, lab in ((-w/2, 'normal', CY, 'GARCH-normal'), (w/2, 't', OR, 'GARCH-t')):
    vals = [sub[(sub.ticker == t) & (sub.block == b) & (sub.model == m)]['breach_rate_%'].iloc[0] for t, b in groups]
    ax.bar(x + off, vals, w, color=col, label=lab)
ax.axhline(1, color='white', ls='--', lw=1.2, label='nominal 1%')
ax.set_xticks(x)
ax.set_xticklabels([f'{t}\n{b[-1]}' for t, b in groups], color='white', fontsize=8.5)
ax.set_ylabel('99% VaR breach rate (%)', color='#9ca3af')
ax.set_title('Parts 2-3: VaR coverage by 750-day block', color='white', fontsize=11)
ax.legend(facecolor='#161b22', edgecolor='#374151', labelcolor='white', fontsize=8.5)

# (b) Part 4
ax = axes[0, 1]
order = list(p4.sort_values('garch_over_ml')['ticker'])
for i, tk in enumerate(order):
    s = p4b[p4b.ticker == tk]
    ax.scatter(s['garch_over_ml'], [i] * len(s), color=GY, s=24, zorder=2)
    ax.scatter(p4.set_index('ticker').loc[tk, 'garch_over_ml'], i, color=CY, s=80, zorder=3)
ax.axvline(1, color='white', ls='--', lw=1.2)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, color='white', fontsize=8.5)
ax.set_xlabel('GARCH RMSE / ML RMSE (below 1: GARCH better)', color='#9ca3af', fontsize=9)
ax.set_title('Part 4: GARCH vs. Gradient Boosting, ten assets', color='white', fontsize=11)

# (c) Part 5
ax = axes[0, 2]
o5 = p5.sort_values('dAIC_gjr')
x = np.arange(len(o5))
ax.bar(x - 0.2, o5['dAIC_gjr'], 0.4, color=CY, label='GJR')
ax.bar(x + 0.2, o5['dAIC_egarch'], 0.4, color=OR, label='EGARCH')
ax.axhline(0, color='white', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(o5['ticker'], color='white', rotation=40, ha='right', fontsize=8.5)
ax.set_ylabel('Change in AIC vs. symmetric GARCH', color='#9ca3af', fontsize=9)
ax.set_title('Part 5: leverage models, full-sample fit', color='white', fontsize=11)
ax.legend(facecolor='#161b22', edgecolor='#374151', labelcolor='white', fontsize=8.5, loc='lower right')

# (d) Part 6
ax = axes[1, 0]
models = ['naive', 'har', 'gbr_base', 'gbr_rich', 'gbr_rich_vix', 'ridge_rich', 'gru']
names = ['Naive', 'HAR', 'GBR\n(Part 1)', 'GBR\nrich', 'GBR\nrich+VIX', 'Ridge\nrich', 'GRU']
vals = [p6[m].mean() for m in models]
cols_ = [GY, GR, OR, OR, OR, CY, CY]
bars = ax.bar(range(len(models)), vals, color=cols_)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.008, f'{v:.2f}', ha='center', fontsize=8.5, color='white')
ax.axhline(1, color='white', ls='--', lw=1.2)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(names, color='white', fontsize=8.5)
ax.set_ylim(0.8, 1.25)
ax.set_ylabel('Mean RMSE / GARCH RMSE, six assets', color='#9ca3af', fontsize=9)
ax.set_title('Part 6: ML variants against GARCH', color='white', fontsize=11)

# (e) Part 7
ax = axes[1, 1]
cols7 = ['har', 'ridge_rich', 'gru', 'garch + har', 'garch + ridge', 'garch + gru', 'garch + ridge + gru', 'har + ridge + gru (no garch)']
names7 = ['HAR', 'Ridge', 'GRU', 'G+HAR', 'G+Ridge', 'G+GRU', 'G+Ridge\n+GRU', 'Ridge+GRU\n+HAR']
vals = [p7[c].mean() for c in cols7]
bars = ax.bar(range(len(cols7)), vals, color=[GR, CY, CY, PU, PU, PU, PU, GY])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.004, f'{v:.3f}', ha='center', fontsize=8, color='white')
ax.axhline(1, color='white', ls='--', lw=1.2)
ax.set_xticks(range(len(cols7)))
ax.set_xticklabels(names7, color='white', fontsize=8)
ax.set_ylim(0.88, 1.02)
ax.set_ylabel('Mean RMSE / GARCH RMSE, six assets', color='#9ca3af', fontsize=9)
ax.set_title('Part 7: single models and averages', color='white', fontsize=11)

# (f) Part 8
ax = axes[1, 2]
h = p8.groupby('horizon')[['naive_over_garch', 'har_over_garch', 'gbr_over_garch']].mean()
for col, c, lab in (('naive_over_garch', GY, 'Naive'), ('har_over_garch', GR, 'HAR'), ('gbr_over_garch', OR, 'Gradient Boosting')):
    ax.plot(h.index, h[col], marker='o', color=c, lw=2, label=lab)
ax.axhline(1, color='white', ls='--', lw=1.2)
ax.set_xticks(list(h.index))
ax.set_xlabel('Forecast horizon (days)', color='#9ca3af')
ax.set_ylabel('Mean RMSE / GARCH RMSE, six assets', color='#9ca3af', fontsize=9)
ax.set_title('Part 8: by forecast horizon', color='white', fontsize=11)
ax.legend(facecolor='#161b22', edgecolor='#374151', labelcolor='white', fontsize=8.5)
plt.tight_layout()
plt.savefig('chart_summary.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved chart_summary.png')